In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#convert CVAT to COCO format
import json
import os

CVAT_JSON_PATH = '<CVAT_JSON_PATH>'
OUTPUT_COCO_PATH = '<OUTPUT_COCO_PATH>'
with open(CVAT_JSON_PATH, 'rb') as f:
    raw = f.read()

json_start = raw.index(b'{')
data = json.loads(raw[json_start:].decode('utf-8', errors='replace'))

cvat_labels = data['categories']['label']['labels']

coco_categories = []
label_name_to_id = {}

for idx, label in enumerate(cvat_labels):
    cat_id = idx + 1
    coco_categories.append({
        'id': cat_id,
        'name': label['name'],
        'supercategory': label.get('parent', '') or 'object'
    })
    label_name_to_id[label['name']] = cat_id

coco_images = []
coco_annotations = []
annotation_id = 1

for item in data['items']:
    frame_str = item['id']
    frame_num = int(frame_str.replace('frame_', ''))
    image_id = frame_num

    coco_images.append({
        'id': image_id,
        'file_name': f"frame_{frame_num:06d}.jpg",
        'width': 0,
        'height': 0,
    })

    for ann in item.get('annotations', []):
        if ann.get('type') != 'bbox':
            continue

        track_id = ann['attributes'].get('track_id', -1)
        occluded = ann['attributes'].get('occluded', False)

        x, y, w, h = ann['bbox']

        cvat_label_id = ann.get('label_id', 0)
        category_id = cvat_label_id + 1

        coco_annotations.append({
            'id': annotation_id,
            'image_id': image_id,
            'category_id': category_id,
            'bbox': [round(x, 4), round(y, 4), round(w, 4), round(h, 4)],
            'area': round(w * h, 4),
            'iscrowd': 0,
            'track_id': track_id,
            'occluded': occluded,
        })
        annotation_id += 1

coco_output = {
    'info': {
        'description': 'Converted from CVAT JSON',
        'source': CVAT_JSON_PATH,
    },
    'licenses': [],
    'categories': coco_categories,
    'images': coco_images,
    'annotations': coco_annotations,
}

os.makedirs(os.path.dirname(OUTPUT_COCO_PATH), exist_ok=True)
with open(OUTPUT_COCO_PATH, 'w') as f:
    json.dump(coco_output, f, indent=2)

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision --quiet
!pip install transformers pillow tqdm --quiet
!pip install loguru
!pip install thop
!pip install lap
import torch
print("CUDA available:", torch.cuda.is_available())


In [ ]:
!git clone https://github.com/ifzhang/ByteTrack.git
%cd ByteTrack
!pip3 install -r requirements.txt
!python3 setup.py develop
!pip3 install cython
!pip3 install 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
!pip3 install cython_bbox

In [ ]:
#DETR + Byte track code
import cv2
import torch
import numpy as np
np.float = float
import math
from tqdm import tqdm
from PIL import Image
from transformers import DetrForObjectDetection, DetrImageProcessor
from yolox.tracker.byte_tracker import BYTETracker
from collections import namedtuple, defaultdict
import json
import gc

# === STEP 3: Set up paths on Google Drive ===
VIDEO_PATH   = "<Video_path>"
MODEL_DIR    = "<Model_path>"
OUTPUT_ROOT  = "<Output_path>"

import os
os.makedirs(OUTPUT_ROOT, exist_ok=True)

PART_R = 1
PART_C = 0
assert os.path.isfile(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
assert os.path.isdir(MODEL_DIR), f"Model directory not found: {MODEL_DIR}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = DetrForObjectDetection.from_pretrained(MODEL_DIR, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(MODEL_DIR)
model.eval()

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (a[2] - a[0]) * (a[3] - a[1])
    area2 = (b[2] - b[0]) * (b[3] - b[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def non_max_suppression(boxes, scores, iou_thr=0.20):
    if not boxes:
        return [], []
    idxs = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep_b, keep_s = [], []
    while idxs:
        i = idxs.pop(0)
        keep_b.append(boxes[i])
        keep_s.append(scores[i])
        idxs = [j for j in idxs if iou(boxes[i], boxes[j]) < iou_thr]
    return keep_b, keep_s

# 4.5 Utility to create per-part directories
def make_part_dirs(root, r, c):
    part_folder = os.path.join(root, f"part_{r}_{c}")
    os.makedirs(part_folder, exist_ok=True)

    debug_dir = os.path.join(part_folder, "debug_frames_inference")
    os.makedirs(debug_dir, exist_ok=True)

    output_video_path = os.path.join(part_folder, f"custom_output_video_part{r}_{c}.mp4")
    tracking_json     = os.path.join(part_folder, f"tracking_summary_part{r}_{c}.json")
    return debug_dir, output_video_path, tracking_json

# 4.6 Read video metadata (once)
tmp_cap = cv2.VideoCapture(VIDEO_PATH)
frame_rate   = int(tmp_cap.get(cv2.CAP_PROP_FPS))
W            = int(tmp_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(tmp_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(tmp_cap.get(cv2.CAP_PROP_FRAME_COUNT))
tmp_cap.release()

print(f"Video resolution: {W}×{H}, FPS: {frame_rate}, Total frames: {total_frames}")

num_seconds = math.ceil(total_frames / frame_rate)
total_slots = num_seconds * 3
col_width  = W  // 3
row_height = H  // 3

r = PART_R
c = PART_C
crop_x1 = c * col_width
crop_x2 = W if c == 2 else (c + 1) * col_width
crop_y1 = r * row_height
crop_y2 = H if r == 2 else (r + 1) * row_height

cw = crop_x2 - crop_x1
ch = crop_y2 - crop_y1
debug_dir, output_video_path, tracking_json = make_part_dirs(OUTPUT_ROOT, r, c)
cap = cv2.VideoCapture(VIDEO_PATH)

Args = namedtuple('Args', ['track_thresh', 'match_thresh', 'track_buffer', 'frame_rate', 'mot20'])
tracker_args = Args(0.3, 0.2, 100, frame_rate, True)
tracker = BYTETracker(tracker_args)
trajectories     = defaultdict(list)
frame_selections = dict()
tracking_hashmap = defaultdict(lambda: [None] * total_slots)
prev_ids         = set()

fourcc    = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter(output_video_path, fourcc, frame_rate, (cw, ch))

print(f"\n▶ Processing ONE part ({r},{c}) → crop = x[{crop_x1}:{crop_x2}]  y[{crop_y1}:{crop_y2}]")
for fidx in tqdm(range(total_frames), desc=f"Part ({r},{c})"):
    ret, frame = cap.read()
    if not ret:
        break

    crop_frame = frame[crop_y1:crop_y2, crop_x1:crop_x2]
    pil = Image.fromarray(cv2.cvtColor(crop_frame, cv2.COLOR_BGR2RGB))

    enc = processor(images=pil, return_tensors="pt").to(device)
    with torch.no_grad():
        outp = model(**enc)

    logits   = outp.logits.softmax(-1)[0].cpu()
    raw_bx   = outp.pred_boxes[0].cpu().numpy()
    scores, _ = logits.max(-1)
    mask       = scores > 0.3

    det_boxes, det_scores = [], []
    for (cx_n, cy_n, w_n, h_n), s in zip(raw_bx[mask], scores[mask].numpy()):
        x1 = (cx_n - w_n / 2) * cw
        y1 = (cy_n - h_n / 2) * ch
        x2 = x1 + w_n * cw
        y2 = y1 + h_n * ch
        det_boxes.append((x1, y1, x2, y2))
        det_scores.append(float(s))
    det_boxes, det_scores = non_max_suppression(det_boxes, det_scores)

    if det_boxes:
        det_arr = np.array(
            [[bx[0], bx[1], bx[2], bx[3], sc] for bx, sc in zip(det_boxes, det_scores)],
            dtype=np.float32
        )
    else:
        det_arr = np.empty((0, 5), dtype=np.float32)


    tracks = tracker.update(det_arr, (ch, cw), (ch, cw))
    curr_ids, curr_boxes = set(), dict()

    for t in tracks:
        tid = t.track_id
        bb  = t.tlbr
        x1, y1, x2, y2 = bb
        cx_c, cy_c = (x1 + x2) / 2, (y1 + y2) / 2
        curr_ids.add(tid)
        curr_boxes[tid] = bb
        trajectories[tid].append((fidx, cx_c, cy_c))
        color = (0,255,0) if tid in prev_ids else (0,0,255)
        cv2.rectangle(crop_frame,
                      (int(x1), int(y1)),
                      (int(x2), int(y2)),
                      color, 2)
        cv2.putText(
            crop_frame,
            f"ID {tid}",
            (int(x1), int(y1) - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2
        )

    prev_ids = curr_ids.copy()
    debug_path = os.path.join(debug_dir, f"frame_{fidx:06d}.jpg")
    cv2.imwrite(debug_path, crop_frame)
    if fidx % frame_rate in [0, 20, 40]:
        second_index = fidx // frame_rate
        slot_offset  = [0, 20, 40].index(fidx % frame_rate)
        relative_idx = second_index * 3 + slot_offset

        frame_selections[fidx] = {}
        disp_frame = crop_frame.copy()
        cv2.putText(
            disp_frame,
            f"Frame {fidx}",
            (cw - 160, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            2
        )

        for tid in curr_ids:
            rounded_box = [round(float(x), 2) for x in curr_boxes[tid]]
            tracking_hashmap[tid][relative_idx] = rounded_box
            frame_selections[fidx][tid] = curr_boxes[tid]

        for _ in range(frame_rate):
            out_video.write(disp_frame)

cap.release()
out_video.release()
cv2.destroyAllWindows()

with open(tracking_json, "w") as f:
    json.dump(tracking_hashmap, f, indent=2)

print(f" Done part ({r},{c})")
print(f"   Debug frames → {debug_dir}")
print(f"   Output video → {output_video_path}")
print(f"   Tracking JSON → {tracking_json}")

# --- Cleanup to release RAM and GPU ---
del cap, tracker, trajectories, frame_selections, tracking_hashmap, prev_ids, out_video
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Part processing complete. Files are under:", OUTPUT_ROOT)


In [ ]:
#DETR detection on full frame
import os
import cv2
import torch
import torchvision
import numpy as np
np.float = float
import math
from tqdm import tqdm
from PIL import Image
from transformers import DetrForObjectDetection, DetrImageProcessor
from collections import namedtuple, defaultdict
import json

IMAGE_PATH = '<IMAGE_PATH>'
MODEL_DIR = '<MODEL_DIR>'
OUTPUT_ROOT = '<OUTPUT_ROOT>'

CONFIDENCE_THRESHOLD = 0.85
IOU_THRESHOLD = 0.2

os.makedirs(OUTPUT_ROOT, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DetrForObjectDetection.from_pretrained(MODEL_DIR, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(MODEL_DIR)
model.eval()

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (a[2] - a[0]) * (a[3] - a[1])
    area2 = (b[2] - b[0]) * (b[3] - b[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def non_max_suppression(boxes, scores, iou_thr):
    if not boxes:
        return [], []
    idxs = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep_b, keep_s = [], []
    while idxs:
        i = idxs.pop(0)
        keep_b.append(boxes[i])
        keep_s.append(scores[i])
        idxs = [j for j in idxs if iou(boxes[i], boxes[j]) < iou_thr]
    return keep_b, keep_s

def process_single_image_grid(image_path, grid_rows=4, grid_cols=4):
    if not os.path.exists(image_path):
        return

    full_image = cv2.imread(image_path)
    if full_image is None:
        return

    H, W, _ = full_image.shape
    col_width = W // grid_cols
    row_height = H // grid_rows

    for r in range(grid_rows):
        for c in range(grid_cols):
            crop_x1 = c * col_width
            crop_x2 = W if c == (grid_cols - 1) else (c + 1) * col_width
            crop_y1 = r * row_height
            crop_y2 = H if r == (grid_rows - 1) else (r + 1) * row_height

            cw, ch = crop_x2 - crop_x1, crop_y2 - crop_y1
            crop_img = full_image[crop_y1:crop_y2, crop_x1:crop_x2].copy()

            pil_img = Image.fromarray(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
            inputs = processor(images=pil_img, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model(**inputs)

            logits = outputs.logits.softmax(-1)[0].cpu()
            pred_boxes = outputs.pred_boxes[0].cpu().numpy()

            scores, labels = logits.max(-1)
            keep = scores > CONFIDENCE_THRESHOLD

            boxes_px = []
            scores_px = []

            for (cx, cy, w, h), score in zip(pred_boxes[keep], scores[keep].numpy()):
                x1 = (cx - w/2) * cw
                y1 = (cy - h/2) * ch
                x2 = x1 + w * cw
                y2 = y1 + h * ch
                boxes_px.append((x1, y1, x2, y2))
                scores_px.append(float(score))

            final_boxes, final_scores = non_max_suppression(boxes_px, scores_px, IOU_THRESHOLD)
            json_data = []

            for box, score in zip(final_boxes, final_scores):
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(crop_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(crop_img, f"{score:.2f}", (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

                json_data.append({
                    "bbox": [x1, y1, x2, y2],
                    "score": float(score)
                })

            part_dir = os.path.join(OUTPUT_ROOT, f"part_{r}_{c}")
            os.makedirs(part_dir, exist_ok=True)

            save_img_path = os.path.join(part_dir, f"result_img_{r}_{c}.jpg")
            cv2.imwrite(save_img_path, crop_img)

            save_json_path = os.path.join(part_dir, f"result_data_{r}_{c}.json")
            with open(save_json_path, 'w') as f:
                json.dump(json_data, f, indent=2)

process_single_image_grid(IMAGE_PATH)